# Order-support agent: a router workflow with a ReAct branch and human approval

This notebook is the worked example of [Lesson 03](../03-building-agents-orchestration-and-memory.md). It builds the order-support assistant used throughout the lecture as a **graph**: typed state, nodes that update it, conditional edges that route between them, a ReAct loop for order questions, and a pause for human approval before a return is created.

It is adapted from the `mail_sorting` notebook of the [Hugging Face Agents Course](https://huggingface.co/agents-course/notebooks/blob/main/unit2/langgraph/mail_sorting.ipynb) (Apache-2.0), which classifies emails and drafts replies with LangGraph.

**Running it.** Set `USE_FAKE_LLM = False` and the environment variables below to use a real Azure OpenAI deployment. With `USE_FAKE_LLM = True` (the default) a small scripted stand-in replaces the model, so the whole graph runs offline and you can study the control flow before spending tokens.

In [1]:
! pip install -U langgraph langchain-openai
import os, hashlib, re
from typing import Annotated, Literal, Optional, TypedDict

from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition, InjectedState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

USE_FAKE_LLM = True   # set to False to call Azure OpenAI

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


  Using cached langgraph_prebuilt-1.1.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/575.5 kB ? eta -:-

ModuleNotFoundError: No module named 'langchain_core'

## 1. The model

The real model is an Azure OpenAI deployment, created from environment variables so that no secret is written in the notebook. `temperature=0` keeps classification stable between runs. The deployment name goes in `azure_deployment`.

In [ ]:
def make_llm():
    from langchain_openai import AzureChatOpenAI
    return AzureChatOpenAI(
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=os.environ.get("OPENAI_API_VERSION", "2024-10-21"),
        azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        temperature=0,
    )

## 2. Data and tools

The data is a tiny in-memory stand-in for the order system. The three tools follow the design rules of Lesson 01: one tool per question, descriptions that say what the tool returns and which sibling to use instead, and no argument for anything the program already knows.

Note the `state: Annotated[dict, InjectedState]` parameter of `orders_get`. The model never sees it and cannot fill it: the graph injects the current state, so the **customer identity comes from the session**, and a customer cannot read someone else's order by guessing a number. `returns_create` is not given to the model at all; only the approval node calls it.

In [ ]:
CUSTOMERS = {"jane@example.com": {"customer_id": "C-1", "name": "Jane Doe", "recent_orders": ["A-10492", "A-10311"]}}
ORDERS = {
    "A-10492": {"customer_id": "C-1", "item": "Kettle", "status": "delivered", "delivered": "2026-10-03", "return_possible": True},
    "A-10311": {"customer_id": "C-1", "item": "Toaster", "status": "delivered", "delivered": "2026-06-14", "return_possible": False},
    "A-20001": {"customer_id": "C-9", "item": "Lamp", "status": "shipped", "delivered": None, "return_possible": False},
}
RETURNS = {}  # idempotency key -> return id


@tool
def customers_search(email: str) -> dict:
    "Find a customer by email address. Returns the name and the three most recent order numbers. Use orders_get for the status of an order."
    c = CUSTOMERS.get(email.lower())
    return c if c else {"found": False, "note": "No customer with this email."}


@tool
def orders_get(order_id: str, state: Annotated[dict, InjectedState]) -> dict:
    "Fetch one order by its number, for example 'A-10492'. Returns item, status, delivery date and whether a return is still possible. Use customers_search to find a customer's orders."
    order = ORDERS.get(order_id.strip().upper())
    if order is None or order["customer_id"] != state["customer_id"]:   # ownership checked in code
        return {"found": False, "note": f"No order {order_id} for this customer."}
    return {"found": True, "order_id": order_id.upper(), **{k: v for k, v in order.items() if k != "customer_id"}}


def returns_create(order_id: str, customer_id: str) -> str:
    "Mutating: never exposed to the model. Idempotent: the same order and customer create one return."
    key = hashlib.sha256(f"{customer_id}:{order_id}".encode()).hexdigest()[:12]
    RETURNS.setdefault(key, f"R-{len(RETURNS) + 1:04d}")
    return RETURNS[key]


AGENT_TOOLS = [customers_search, orders_get]

## 3. State: the contract between nodes

Every node reads the state and returns a partial update. `messages` uses the `add_messages` reducer, so updates are **appended** instead of replacing the list, which is how the ReAct loop accumulates its thoughts, tool calls and observations (the agent's short-term memory). The other fields are overwritten.

In [ ]:
class SupportState(TypedDict):
    customer_id: str                        # from the authenticated session, never from the model
    message: str                            # the customer's request
    category: Optional[str]                 # set by classify
    order_id: Optional[str]                 # extracted by classify when present
    messages: Annotated[list, add_messages] # ReAct transcript, appended by the reducer
    draft: Optional[str]                    # reply proposed to the customer
    return_id: Optional[str]                # set only after approval
    outcome: Optional[str]                  # how the run ended, for logging


class Classification(BaseModel):
    category: Literal["order_status", "return_request", "other", "spam"] = Field(description="What the customer wants")
    order_id: Optional[str] = Field(default=None, description="Order number such as A-10492 if the message contains one")

## 4. A scripted stand-in for offline runs

When `USE_FAKE_LLM` is true, these three small classes replace the model. They return the same types a real model would (a `Classification`, an `AIMessage` with tool calls, an `AIMessage` with text), so the graph code is identical in both modes. They are deliberately simple: keyword rules for classification and a fixed two-step plan for the order agent.

In [ ]:
class FakeClassifier:
    def invoke(self, prompt):
        text = prompt[-1].content.lower() if isinstance(prompt, list) else str(prompt).lower()
        m = re.search(r"a-\d{5}", text)
        oid = m.group(0).upper() if m else None
        if any(w in text for w in ["crypto", "winner", "click here"]):
            return Classification(category="spam")
        if any(w in text for w in ["return", "broken", "refund"]):
            return Classification(category="return_request", order_id=oid)
        if any(w in text for w in ["where is", "status", "order"]):
            return Classification(category="order_status", order_id=oid)
        return Classification(category="other")


class FakeToolCaller:
    "First turn: request orders_get. After an observation: answer from it."
    def invoke(self, messages):
        results = [m for m in messages if isinstance(m, ToolMessage)]
        if not results:
            oid = re.search(r"A-\d{5}", messages[-1].content, re.I).group(0).upper()
            return AIMessage(content="", tool_calls=[{"name": "orders_get", "args": {"order_id": oid}, "id": "call_1", "type": "tool_call"}])
        return AIMessage(content=f"Here is what I found: {results[-1].content}")


class FakeWriter:
    def invoke(self, messages):
        return AIMessage(content="Draft: " + messages[-1].content[:160])


if USE_FAKE_LLM:
    classifier, agent_llm, writer = FakeClassifier(), FakeToolCaller(), FakeWriter()
else:
    llm = make_llm()
    classifier = llm.with_structured_output(Classification)   # enumeration, not substring matching
    agent_llm = llm.bind_tools(AGENT_TOOLS)
    writer = llm

## 5. Nodes

Each node is a plain function from state to a partial update.

| Node | Kind | What it does |
| :-- | :-- | :-- |
| `classify` | Model call, structured | Sets `category` and `order_id`, and seeds the transcript for the agent |
| `order_agent` | Model call with tools | One ReAct step: a thought plus a tool request, or a final answer |
| `tools` | Code | Runs the requested tool (prebuilt `ToolNode`) and appends the observation |
| `check_return` | Code | Looks the order up deterministically and decides eligibility |
| `approve_return` | Code, pauses | Shows the draft to a person with `interrupt` and creates the return only on approval |
| `reply` | Model call | Drafts a general reply |
| `discard` | Code | Ends spam without spending a model call |

In [ ]:
SYSTEM = ("You are the order-support assistant of a retail chain. Answer only about the customer's own orders. "
          "Use the tools to look facts up; never guess a status or a date.")


def classify(state: SupportState):
    c = classifier.invoke([SystemMessage("Classify the customer message."), HumanMessage(state["message"])])
    update = {"category": c.category, "order_id": c.order_id}
    if c.category == "order_status":
        update["messages"] = [SystemMessage(SYSTEM), HumanMessage(state["message"])]
    return update


def order_agent(state: SupportState):
    return {"messages": [agent_llm.invoke(state["messages"])]}


def check_return(state: SupportState):
    order = orders_get.func(state["order_id"] or "", state)   # same tool, called by code with the session identity
    if not order.get("found") or not order.get("return_possible"):
        return {"draft": f"Sorry, order {state['order_id']} cannot be returned: {order.get('note', 'the return window has closed')}.", "outcome": "return_refused"}
    text = writer.invoke([SystemMessage("Write a short, friendly reply proposing a return."),
                          HumanMessage(f"Customer message: {state['message']}. Order: {order}")]).content
    return {"draft": text}


def approve_return(state: SupportState):
    decision = interrupt({"action": "returns_create", "order_id": state["order_id"], "draft": state["draft"]})
    if decision != "approve":
        return {"outcome": "return_rejected_by_reviewer"}
    return {"return_id": returns_create(state["order_id"], state["customer_id"]), "outcome": "return_created"}


def reply(state: SupportState):
    text = writer.invoke([SystemMessage(SYSTEM), HumanMessage(state["message"])]).content
    return {"draft": text, "outcome": "replied"}


def discard(state: SupportState):
    return {"outcome": "spam_discarded"}

## 6. Edges: the router and the loop

`route` is the router of Lesson 02: a plain function that reads the state and names the next node. The order branch is a ReAct loop: `tools_condition` sends the agent's message to `tools` when it contains a tool request and to the end otherwise, and `tools` always returns to the agent. The return branch only reaches approval when the order is eligible.

The checkpointer stores the state after every step. It is what lets the graph **pause** at the interrupt and **resume** later, and it is the agent's memory for this conversation thread.

In [ ]:
def route(state: SupportState) -> str:
    return state["category"]


def after_check(state: SupportState) -> str:
    return "refused" if state.get("outcome") == "return_refused" else "eligible"


g = StateGraph(SupportState)
for name, fn in [("classify", classify), ("order_agent", order_agent), ("tools", ToolNode(AGENT_TOOLS)),
                 ("check_return", check_return), ("approve_return", approve_return),
                 ("reply", reply), ("discard", discard)]:
    g.add_node(name, fn)

g.add_edge(START, "classify")
g.add_conditional_edges("classify", route, {"order_status": "order_agent", "return_request": "check_return",
                                            "other": "reply", "spam": "discard"})
g.add_conditional_edges("order_agent", tools_condition, {"tools": "tools", END: END})
g.add_edge("tools", "order_agent")
g.add_conditional_edges("check_return", after_check, {"eligible": "approve_return", "refused": END})
for n in ["approve_return", "reply", "discard"]:
    g.add_edge(n, END)

app = g.compile(checkpointer=InMemorySaver())
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	order_agent(order_agent)
	tools(tools)
	check_return(check_return)
	approve_return(approve_return)
	reply(reply)
	discard(discard)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	check_return -. &nbsp;refused&nbsp; .-> __end__;
	check_return -. &nbsp;eligible&nbsp; .-> approve_return;
	classify -. &nbsp;return_request&nbsp; .-> check_return;
	classify -. &nbsp;spam&nbsp; .-> discard;
	classify -. &nbsp;order_status&nbsp; .-> order_agent;
	classify -. &nbsp;other&nbsp; .-> reply;
	order_agent -.-> __end__;
	order_agent -.-> tools;
	tools --> order_agent;
	approve_return --> __end__;
	discard --> __end__;
	reply --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 7. Running three requests

`recursion_limit` is the step budget of Lesson 02: if the ReAct loop does not finish within it, the run stops with an error instead of looping. Each request gets its own `thread_id`, which is how the checkpointer keeps conversations apart. Streaming with `stream_mode="updates"` prints each node as it finishes, which is the trajectory you would trace in production (Lesson 05).

In [ ]:
def show(update):
    "Compact view of a state update: each message becomes one short line."
    if not isinstance(update, dict):
        return "PAUSED for human approval"
    parts = []
    for k, v in update.items():
        if k == "messages":
            for m in v:
                calls = getattr(m, "tool_calls", None)
                parts.append(f"{type(m).__name__}: " + (f"tool call {calls[0]['name']}({calls[0]['args']})" if calls else str(m.content)[:80]))
        else:
            parts.append(f"{k}={str(v)[:80]}")
    return " | ".join(parts)


def run(message, thread, customer_id="C-1"):
    config = {"configurable": {"thread_id": thread}, "recursion_limit": 10}
    print(f"\n=== {message}")
    for step in app.stream({"customer_id": customer_id, "message": message}, config, stream_mode="updates"):
        for node, update in step.items():
            print(f"  {node:15} {show(update)}")
    return config


_ = run("Where is my order A-10492?", "t1")
_ = run("Congratulations, you are a winner, click here for free crypto", "t2")
cfg = run("My kettle from order A-10492 arrived broken, I want to return it", "t3")


=== Where is my order A-10492?
  classify        category=order_status | order_id=A-10492 | SystemMessage: You are the order-support assistant of a retail chain. Answer only about the cus | HumanMessage: Where is my order A-10492?
  order_agent     AIMessage: tool call orders_get({'order_id': 'A-10492'})
  tools           ToolMessage: {"found": true, "order_id": "A-10492", "item": "Kettle", "status": "delivered", 
  order_agent     AIMessage: Here is what I found: {"found": true, "order_id": "A-10492", "item": "Kettle", "

=== Congratulations, you are a winner, click here for free crypto
  classify        category=spam | order_id=None
  discard         outcome=spam_discarded

=== My kettle from order A-10492 arrived broken, I want to return it
  classify        category=return_request | order_id=A-10492
  check_return    draft=Draft: Customer message: My kettle from order A-10492 arrived broken, I want to 
  __interrupt__   PAUSED for human approval


The third run stopped at `approve_return`: the graph is paused, its state saved by the checkpointer. A person (or a review screen) inspects the pending action and resumes the same thread with a decision. Calling `returns_create` again for the same order shows the idempotency key at work: one return, not two.

In [ ]:
print("Pending:", app.get_state(cfg).tasks[0].interrupts[0].value)
result = app.invoke(Command(resume="approve"), cfg)
print("Outcome:", result["outcome"], "| return id:", result["return_id"])
print("Called again:", returns_create("A-10492", "C-1"), "| returns in the system:", RETURNS)

Pending: {'action': 'returns_create', 'order_id': 'A-10492', 'draft': "Draft: Customer message: My kettle from order A-10492 arrived broken, I want to return it. Order: {'found': True, 'order_id': 'A-10492', 'item': 'Kettle', 'status': 'd"}
Outcome: return_created | return id: R-0001
Called again: R-0001 | returns in the system: {'d2c172297d60': 'R-0001'}


## 8. A request the agent must refuse

Order `A-20001` belongs to another customer. The model can ask for it, but `orders_get` checks ownership against the identity in the state, so the lookup fails and nothing leaks.

In [ ]:
_ = run("Where is my order A-20001?", "t4")


=== Where is my order A-20001?
  classify        category=order_status | order_id=A-20001 | SystemMessage: You are the order-support assistant of a retail chain. Answer only about the cus | HumanMessage: Where is my order A-20001?
  order_agent     AIMessage: tool call orders_get({'order_id': 'A-20001'})
  tools           ToolMessage: {"found": false, "note": "No order A-20001 for this customer."}
  order_agent     AIMessage: Here is what I found: {"found": false, "note": "No order A-20001 for this custom


## What to try next

| Exercise | What it teaches |
| :-- | :-- |
| Set `USE_FAKE_LLM = False` and run with your Azure deployment | How a real model fills the same graph, and where it differs from the script |
| Resume thread `t3` with `"reject"` on a fresh run | The rejected path and the `outcome` field used for logging |
| Lower `recursion_limit` to 2 and ask for an order status | What a step budget does to a ReAct loop |
| Add a `billing` category and a node | Extending the router without touching the other branches |
| Add tracing (Lesson 05) | Seeing each node, model call and tool call as a span with tokens and latency |